In [ ]:
import numpy as np
import pandas as pd

# Set seed for reproducible numbers
np.random.seed(42)

# 1. Generate dates across 1 year (365 days)
dates = pd.date_range(start="2025-01-01", periods=365, freq="D")

# 2. Define 10 realistic products across categories
products = [
    {"Product_ID": "SKU_101", "Category": "Electronics", "Base_Price": 12000, "Base_Demand": 25},
    {"Product_ID": "SKU_102", "Category": "Electronics", "Base_Price": 4500, "Base_Demand": 40},
    {"Product_ID": "SKU_103", "Category": "Electronics", "Base_Price": 1500, "Base_Demand": 85},
    {"Product_ID": "SKU_201", "Category": "Apparel", "Base_Price": 1800, "Base_Demand": 60},
    {"Product_ID": "SKU_202", "Category": "Apparel", "Base_Price": 999, "Base_Demand": 120},
    {"Product_ID": "SKU_203", "Category": "Apparel", "Base_Price": 2499, "Base_Demand": 30},
    {"Product_ID": "SKU_301", "Category": "Home & Kitchen", "Base_Price": 3200, "Base_Demand": 35},
    {"Product_ID": "SKU_302", "Category": "Home & Kitchen", "Base_Price": 750, "Base_Demand": 150},
    {"Product_ID": "SKU_401", "Category": "Personal Care", "Base_Price": 499, "Base_Demand": 210},
    {"Product_ID": "SKU_402", "Category": "Personal Care", "Base_Price": 850, "Base_Demand": 95},
]

records = []

for d in dates:
    # Marketing Promotion occurs randomly roughly 15% of the days
    is_promo = 1 if np.random.rand() < 0.15 else 0
    promo_lift = 1.45 if is_promo == 1 else 1.0  # 45% demand spike during promotion
    
    for p in products:
        # Weekend sales boost
        weekend_boost = 1.2 if d.weekday() >= 5 else 1.0
        
        # Calculate daily sales with standard variation
        mean_demand = p["Base_Demand"] * promo_lift * weekend_boost
        units_sold = int(np.maximum(0, np.random.normal(mean_demand, scale=mean_demand * 0.15)))
        
        # Selling price with promo discount
        price = p["Base_Price"] * (0.90 if is_promo == 1 else 1.0)
        revenue = round(units_sold * price, 2)
        
        records.append({
            "Date": d.strftime("%Y-%m-%d"),
            "Product_ID": p["Product_ID"],
            "Category": p["Category"],
            "Units_Sold": units_sold,
            "Unit_Price": price,
            "Revenue": revenue,
            "On_Promotion": is_promo,
            "Lead_Time_Days": np.random.choice([3, 5, 7])
        })

# Save to a DataFrame and a CSV file
df = pd.DataFrame(records)
df.to_csv("omnichannel_inventory_sales.csv", index=False)

print("Dataset generated successfully!")
print(f"Total Rows: {len(df)}")
display(df.head())

In [ ]:
# --- STEP 2: ABC & XYZ INVENTORY SEGMENTATION ---

# 1. ABC ANALYSIS: Calculate total revenue per SKU and rank them
abc = df.groupby(["Product_ID", "Category"])["Revenue"].sum().reset_index()
abc = abc.sort_values(by="Revenue", ascending=False).reset_index(drop=True)

total_rev = abc["Revenue"].sum()
abc["Revenue_Share_%"] = round((abc["Revenue"] / total_rev) * 100, 2)
abc["Cumulative_Revenue_%"] = abc["Revenue_Share_%"].cumsum()

# Assign ABC classes based on cumulative revenue share
def get_abc_class(cum_pct):
    if cum_pct <= 75:
        return "A"
    elif cum_pct <= 92:
        return "B"
    else:
        return "C"

abc["ABC_Class"] = abc["Cumulative_Revenue_%"].apply(get_abc_class)

# 2. XYZ ANALYSIS: Calculate demand volatility (Coefficient of Variation: CV)
xyz = df.groupby("Product_ID")["Units_Sold"].agg(["mean", "std"]).reset_index()
xyz.columns = ["Product_ID", "Mean_Daily_Demand", "Std_Daily_Demand"]
xyz["CV"] = round(xyz["Std_Daily_Demand"] / xyz["Mean_Daily_Demand"], 3)

# Assign XYZ classes based on CV threshold
def get_xyz_class(cv):
    if cv <= 0.15:
        return "X"  # Highly predictable
    elif cv <= 0.22:
        return "Y"  # Moderate variability
    else:
        return "Z"  # Highly erratic / promotional spikes

xyz["XYZ_Class"] = xyz["CV"].apply(get_xyz_class)

# 3. COMBINE INTO AN ABC-XYZ MATRIX
matrix = pd.merge(abc, xyz, on="Product_ID")
matrix["Segment"] = matrix["ABC_Class"] + matrix["XYZ_Class"]

# Reorder columns cleanly for presentation
final_matrix = matrix[[
    "Product_ID", "Category", "Revenue", "Revenue_Share_%", 
    "Cumulative_Revenue_%", "ABC_Class", "Mean_Daily_Demand", 
    "CV", "XYZ_Class", "Segment"
]]

print("=== COMPLETE ABC-XYZ INVENTORY SEGMENTATION ===")
display(final_matrix)

In [ ]:
# --- STEP 3: SAFETY STOCK & REORDER POINT (ROP) OPTIMIZATION ---

# 1. Calculate Mean and Standard Deviation of Daily Demand directly
stats = df.groupby("Product_ID").agg(
    Mean_Daily_Demand=("Units_Sold", "mean"),
    Std_Daily_Demand=("Units_Sold", "std"),
    Lead_Time_Days=("Lead_Time_Days", "mean")
).reset_index()

# 2. Merge with ABC-XYZ Matrix labels
inv_policy = pd.merge(final_matrix, stats[["Product_ID", "Std_Daily_Demand", "Lead_Time_Days"]], on="Product_ID")

# 3. Parameters (Z = 1.65 for a 95% Cycle Service Level)
Z_SCORE = 1.65  

# 4. Supply Chain Policy Calculations
# Safety Stock = Z * (Std_Daily_Demand) * sqrt(Lead_Time)
inv_policy["Safety_Stock_Units"] = np.ceil(
    Z_SCORE * inv_policy["Std_Daily_Demand"] * np.sqrt(inv_policy["Lead_Time_Days"])
).astype(int)

# Lead Time Demand = Mean_Daily_Demand * Lead_Time
inv_policy["Lead_Time_Demand"] = np.ceil(
    inv_policy["Mean_Daily_Demand"] * inv_policy["Lead_Time_Days"]
).astype(int)

# Reorder Point (ROP) = Lead Time Demand + Safety Stock
inv_policy["Reorder_Point_Units"] = inv_policy["Lead_Time_Demand"] + inv_policy["Safety_Stock_Units"]

# 5. Display the operational inventory policy table
columns_to_show = [
    "Product_ID", "Category", "Segment", "Mean_Daily_Demand", 
    "Lead_Time_Days", "Safety_Stock_Units", "Reorder_Point_Units"
]

print("=== OPERATIONAL INVENTORY REPLENISHMENT POLICIES (95% SERVICE LEVEL) ===")
display(inv_policy[columns_to_show])

In [ ]:
# --- STEP 4: MARKETING PROMOTION LIFT & DEMAND VISUALIZATION ---

import matplotlib.pyplot as plt

# 1. Quantify Promotional Lift across products
promo_analysis = df.groupby(["Product_ID", "Category", "On_Promotion"])["Units_Sold"].mean().unstack().reset_index()
promo_analysis.columns = ["Product_ID", "Category", "Non_Promo_Avg_Demand", "Promo_Avg_Demand"]

# Calculate % Lift created by marketing campaigns
promo_analysis["Promo_Lift_%"] = round(
    ((promo_analysis["Promo_Avg_Demand"] - promo_analysis["Non_Promo_Avg_Demand"]) / promo_analysis["Non_Promo_Avg_Demand"]) * 100, 
    2
)

print("=== MARKETING PROMOTIONAL LIFT BY SKU ===")
display(promo_analysis)

# 2. Visual Plot: Demand spikes for top revenue driver (SKU_101)
sku_sample = df[df["Product_ID"] == "SKU_101"].head(90)  # First quarter (90 days)

plt.figure(figsize=(12, 5))
plt.plot(sku_sample["Date"], sku_sample["Units_Sold"], label="Daily Demand (Units)", color="#1f77b4", linewidth=1.8)

# Highlight marketing promotional days
promo_days = sku_sample[sku_sample["On_Promotion"] == 1]
plt.scatter(promo_days["Date"], promo_days["Units_Sold"], color="red", label="Marketing Promotion Active", zorder=5, s=60)

# Reorder Point threshold line
rop_val = inv_policy[inv_policy["Product_ID"] == "SKU_101"]["Reorder_Point_Units"].values[0]
plt.title("SKU_101 (Electronics): Daily Demand with Promotional Spikes", fontsize=13, fontweight="bold")
plt.xlabel("Date", fontsize=11)
plt.ylabel("Units Sold", fontsize=11)
plt.xticks(sku_sample["Date"][::10], rotation=45)
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend(loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
# --- STEP 5: COMBINE EVERYTHING INTO A FINAL MASTER TABLE ---

# Merge our inventory policies with promotional lift metrics
final_summary = pd.merge(
    inv_policy[[
        "Product_ID", "Category", "Revenue", "Revenue_Share_%", 
        "Cumulative_Revenue_%", "ABC_Class", "XYZ_Class", "Segment",
        "Mean_Daily_Demand", "Lead_Time_Days", "Safety_Stock_Units", "Reorder_Point_Units"
    ]],
    promo_analysis[["Product_ID", "Non_Promo_Avg_Demand", "Promo_Avg_Demand", "Promo_Lift_%"]],
    on="Product_ID"
)

# Save to a final clean CSV file
final_summary.to_csv("inventory_demand_summary.csv", index=False)

print("=== FINAL MASTER INVENTORY & MARKETING SUMMARY ===")
display(final_summary)